# 1 · Returns Generator
Computes simple returns, log returns, the covariance matrix and correlation matrix from the cleaned price matrix.

**All subsequent notebooks work in monthly simple-return space.**  
Log returns are saved for reference but not used in portfolio optimisation.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
sys.path.insert(0, '..')
from utils import ANALYSIS_DIR, PERIODS_PER_YEAR, RF_ANNUAL, RF_MONTHLY

analysis_path = Path(ANALYSIS_DIR)

prices = pd.read_csv(
    analysis_path / "india_equity_prices.csv",
    index_col="Date", parse_dates=True
)

assert not prices.isna().any().any(), "Price matrix still contains NaNs — fix nb 0 first"

# ── Simple percentage returns ─────────────────────────────────────────────────
returns = prices.pct_change().dropna()

# ── Log returns (for reference / normality tests) ────────────────────────────
log_returns = np.log(prices / prices.shift(1)).dropna()

# ── Covariance & correlation (on simple returns) ─────────────────────────────
cov_matrix  = returns.cov()
corr_matrix = returns.corr()

# ── Summary statistics ────────────────────────────────────────────────────────
summary = pd.DataFrame({
    "Mean_Monthly"   : returns.mean(),
    "Std_Monthly"    : returns.std(),
    "Mean_Annual"    : (1 + returns.mean()) ** PERIODS_PER_YEAR - 1,
    "Std_Annual"     : returns.std() * np.sqrt(PERIODS_PER_YEAR),
    "Sharpe_Annual"  : (
        ((1 + returns.mean()) ** PERIODS_PER_YEAR - 1 - RF_ANNUAL)
        / (returns.std() * np.sqrt(PERIODS_PER_YEAR))
    ),
    "Skewness"       : returns.skew(),
    "Kurtosis"       : returns.kurt(),
    "Min_Monthly"    : returns.min(),
    "Max_Monthly"    : returns.max(),
}).round(6)

# ── Save outputs ─────────────────────────────────────────────────────────────
returns.to_csv(analysis_path / "india_equity_returns.csv")
log_returns.to_csv(analysis_path / "india_equity_log_returns.csv")
cov_matrix.to_csv(analysis_path / "india_covariance_matrix.csv")
corr_matrix.to_csv(analysis_path / "india_correlation_matrix.csv")
summary.to_csv(analysis_path / "asset_summary_statistics.csv")

print(f"Periods per year : {PERIODS_PER_YEAR}  (monthly data)")
print(f"Risk-free rate   : {RF_ANNUAL*100:.2f}% p.a.  →  {RF_MONTHLY*100:.4f}% per month")
print(f"\nReturns matrix   : {returns.shape[0]} periods × {returns.shape[1]} assets")
print("\nSaved to analysis/")
summary

Periods per year : 12  (monthly data)
Risk-free rate   : 6.50% p.a.  →  0.5417% per month

Returns matrix   : 85 periods × 9 assets

Saved to analysis/


,Mean_Monthly,Std_Monthly,Mean_Annual,Std_Annual,Sharpe_Annual,Skewness,Kurtosis,Min_Monthly,Max_Monthly
BRTI,0.024407,0.065346,0.335580,0.226365,1.195328,-0.214808,0.828459,-0.179593,0.182315
EICH,0.019384,0.084104,0.259078,0.291345,0.666144,0.107603,0.552730,-0.211200,0.268666
HDBK,0.008132,0.065630,0.102068,0.227350,0.163044,-0.281862,3.761804,-0.268115,0.217394
HLL,0.004900,0.061753,0.060405,0.213919,-0.021482,0.432502,0.317045,-0.145372,0.182374
LART,0.016497,0.076848,0.216948,0.266209,0.570786,-0.613962,3.628496,-0.319190,0.230921
RELI,0.013735,0.075916,0.177859,0.262982,0.429151,0.881577,2.139479,-0.161737,0.316261
SUN,0.019309,0.075991,0.257974,0.263240,0.733073,0.628159,1.608913,-0.141377,0.318337
TISC,0.022490,0.112082,0.305892,0.388265,0.620432,0.114398,1.271829,-0.293686,0.406235
TITN,0.020316,0.082470,0.272965,0.285685,0.727951,-0.721175,1.035583,-0.255719,0.211616
